# TOML

TOML (Tom's Obvious Minimal Language) формат заявлен самим составителем Томом-Вернером Престоном на странице [проекта](https://toml.io/en/v1.0.0) так:

> TOML aims to be a minimal configuration file format that's easy to read due to obvious semantics.
> TOML is designed to map unambiguously to a hash table.
> TOML should be easy to parse into data structures in a wide variety of languages.

Т.е. это минимальный формат описания конфигурационных файлов, который можно однозначно представить в хэш-таблицу (словарь в Python).

### Быстрый разгон

Поддержка TOML была добавлена в стандартную библиотеку Python с версии 3.11 - библиотека [tomllib](https://docs.python.org/3/library/tomllib.html), а для прежних изводов (версий) можно или нужно прибегать к сторонним пакетам вроде [toml](https://pypi.org/project/toml/).

Посмотрим примеры (большей частью из [официальной документации](https://toml.io/en/v1.0.0)).

In [1]:
import tomllib

In [2]:
doc = """
# comment
key = "strval"
int_key_underscored = 123
bool-key-hyphentaed = false
456 = 'int-as-string-key with single-quoted string value'
'' = 'empty keys are allowed yet discouraged'
"""

tomllib.loads(doc)

{'key': 'strval',
 'int_key_underscored': 123,
 'bool-key-hyphentaed': False,
 '456': 'int-as-string-key with single-quoted string value',
 '': 'empty keys are allowed yet discouraged'}

А теперь поиграемся с "точкоразделёнными" (dotted) ключами и кавычками: первые отвечают за словарную вложенность, вторые позволяют задавать сложносоставные имена ключей, например.

In [3]:
doc = """
key = 0b1101
dotted.key = 0o123
fake-answer."combined.single.key" = 0x42
superkey  . subkey = 123_456
"I.am . floating" = 3.14_15
"I-am-exponential" = -2E-2

# infinity
sf1 = inf  # positive infinity
sf2 = +inf # positive infinity
sf3 = -inf # negative infinity

# not a number
sf4 = nan  # actual sNaN/qNaN encoding is implementation-specific
sf5 = +nan # same as `nan`
sf6 = -nan # valid, actual encoding is implementation-specific
"""

tomllib.loads(doc)

{'key': 13,
 'dotted': {'key': 83},
 'fake-answer': {'combined.single.key': 66},
 'superkey': {'subkey': 123456},
 'I.am . floating': 3.1415,
 'I-am-exponential': -0.02,
 'sf1': inf,
 'sf2': inf,
 'sf3': -inf,
 'sf4': nan,
 'sf5': nan,
 'sf6': nan}

Видно, что кавычки задают имя ключа в сыром (raw) виде. Также пробелы меж разделительными точками вполне допустимы, но лучше этим не злоупотреблять, а в "закавыченных" именах они и вовсе не "скорачиваются", а остаются как есть.

Даты, времена и даты-времена.

In [4]:
doc = """
ld1 = 1979-05-27

lt1 = 07:32:00
lt2 = 00:32:00.999999

odt1 = 1979-05-27T07:32:00Z
odt2 = 1979-05-27T00:32:00-07:00
odt3 = 1979-05-27T00:32:00.999999-07:00
odt4 = 1979-05-27 07:32:00Z  # without T which is replaced with a whitespace
"""

tomllib.loads(doc)

{'ld1': datetime.date(1979, 5, 27),
 'lt1': datetime.time(7, 32),
 'lt2': datetime.time(0, 32, 0, 999999),
 'odt1': datetime.datetime(1979, 5, 27, 7, 32, tzinfo=datetime.timezone.utc),
 'odt2': datetime.datetime(1979, 5, 27, 0, 32, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))),
 'odt3': datetime.datetime(1979, 5, 27, 0, 32, 0, 999999, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))),
 'odt4': datetime.datetime(1979, 5, 27, 7, 32, tzinfo=datetime.timezone.utc)}

Массивы (arrays) - данные (элементы, items) в \[квадратных\] скобках. Составляющие массива разделены запятыми, а сам массив может содержать разнородные (гетерогенные) данные, что удобно.

In [5]:
doc = """
integers = [ 1, 2, 3 ]
colors = [ "red", 'yellow', "green" ]
nested_arrays_of_ints = [ [ 1, 2 ], [3, 4, 5] ]
nested_mixed_array = [ [ 1, 2 ], ["a", "b", "c"] ]

# Mixed-type arrays are allowed
numbers = [ 0.1, 0.2, 0.5, 1, 2, 5 , "Foo Bar <foo@example.com>"]
"""

tomllib.loads(doc)

{'integers': [1, 2, 3],
 'colors': ['red', 'yellow', 'green'],
 'nested_arrays_of_ints': [[1, 2], [3, 4, 5]],
 'nested_mixed_array': [[1, 2], ['a', 'b', 'c']],
 'numbers': [0.1, 0.2, 0.5, 1, 2, 5, 'Foo Bar <foo@example.com>']}

### Таблицы

Ранее, играясь с точкоразделёнными ключами, уже было столкновение с таблицами, просто про них было удобно говорить как о словарях для конечного Python потребителя. Но сам формат позволяет разметить документ на явно задаваемы разделы (секции) и здесь таблицы в самую помощь. 

In [6]:
doc = """
[table-1]
key11 = 12

[section-2]
key21 = 21

[super_table."sub.section"]
key = 42
type.name = "spam"

[a.b.c]
# empty table/section/disctionaty
"""

tomllib.loads(doc)

{'table-1': {'key11': 12},
 'section-2': {'key21': 21},
 'super_table': {'sub.section': {'key': 42, 'type': {'name': 'spam'}}},
 'a': {'b': {'c': {}}}}

И можно вспомнить простое "умолчательное" содержимое файла "pyproject.toml" проекта pack1.

In [7]:
doc = """
[project]
name = "pack1"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.14"
dependencies = []
"""

tomllib.loads(doc)

{'project': {'name': 'pack1',
  'version': '0.1.0',
  'description': 'Add your description here',
  'readme': 'README.md',
  'requires-python': '>=3.14',
  'dependencies': []}}

А вот "прелесть" уже из второй главы. Особое внимание на ключи authors/maintainers - это списки словарей, ещё их называют inline tables.

In [8]:
doc = """
[build-system]
requires = ["setuptools ~= 84.0"]  # the frontend should install them automatically
build-backend = "setuptools.build_meta"  # the path to the backend program

[project]
name = "example"
version = "0.0.2"
authors = [
    { name = "Author Name", email = "author@example.com" },
]
maintainers = [
    { name = "Maintainer Name", email = "maintainer@example.com" },
]
description = "A sample Python package v2"
requires-python = "~=3.14.0"
readme = "README.md"
keywords = ["building", "python", "packages"]
# https://pypi.org/classifiers/
classifiers = [
    "Development Status :: 3 - Alpha",
    "Intended Audience :: Developers",
    # SetuptoolsDeprecationWarning: License classifiers are deprecated.
    # "License :: Free For Home Use",
    "Programming Language :: Python :: 3",
    "Programming Language :: Python :: 3.14",
    "Topic :: Education",
]
dependencies = []

[dependency-groups]
test = [
    "pytest>=9.1.1",
]

[tool.setuptools.packages.find]
# The name "example" is a special one for the package.
# Setuptools excludes it by default for a flat-layout packages.
# https://setuptools.pypa.io/en/latest/userguide/package_discovery.html#setuptools.discovery.FlatLayoutPackageFinder.DEFAULT_EXCLUDE
include = ["example*"]
"""

tomllib.loads(doc)

{'build-system': {'requires': ['setuptools ~= 84.0'],
  'build-backend': 'setuptools.build_meta'},
 'project': {'name': 'example',
  'version': '0.0.2',
  'authors': [{'name': 'Author Name', 'email': 'author@example.com'}],
  'maintainers': [{'name': 'Maintainer Name',
    'email': 'maintainer@example.com'}],
  'description': 'A sample Python package v2',
  'requires-python': '~=3.14.0',
  'readme': 'README.md',
  'keywords': ['building', 'python', 'packages'],
  'classifiers': ['Development Status :: 3 - Alpha',
   'Intended Audience :: Developers',
   'Programming Language :: Python :: 3',
   'Programming Language :: Python :: 3.14',
   'Topic :: Education'],
  'dependencies': []},
 'dependency-groups': {'test': ['pytest>=9.1.1']},
 'tool': {'setuptools': {'packages': {'find': {'include': ['example*']}}}}}